## ETL
### Paso 1: Generando el "Dataset Sucio"
Vamos a simular una exportación fallida del sistema de la facultad.

In [ ]:
import pandas as pd
import numpy as np

# Creamos datos intencionalmente sucios (con NaNs y filas duplicadas)
datos_sucios = {
    'Legajo': ['TUP-01', 'TUP-02', 'TUP-03', 'TUP-04', 'TUP-02', 'TUP-05', 'TUP-06'],
    'Nombre': ['Ana', 'Juan', 'Pedro', 'Laura', 'Juan', np.nan, 'Carlos'],
    'Comision': ['A', 'A', 'B', np.nan, 'A', 'C', 'C'],
    'Nota_Final': [8.5, np.nan, 4.0, 9.0, np.nan, 7.5, 6.0]
}

df_crudo = pd.DataFrame(datos_sucios)

print('--- DATOS CRUDOS ---')
display(df_crudo)

--- DATOS CRUDOS ---


,Legajo,Nombre,Comision,Nota_Final
0,TUP-01,Ana,A,8.5
1,TUP-02,Juan,A,NaN
2,TUP-03,Pedro,B,4.0
3,TUP-04,Laura,NaN,9.0
4,TUP-02,Juan,A,NaN
5,TUP-05,NaN,C,7.5
6,TUP-06,Carlos,C,6.0


### Paso 2: Auditoría de Calidad (Lo que pide el TPI)
Antes de limpiar, tenemos que medir la basura.

In [ ]:
print('--- CONTEO DE VALORES NULOS ---')
# isnull() crea una máscara booleana, sum() cuenta los True
nulos_por_columna = df_crudo.isnull().sum()
print(nulos_por_columna) # int64

print('\n--- CONTEO DE DUPLICADOS ---')
duplicados_totales = df_crudo.duplicated().sum()
print('Filas exactamente duplicadas:', duplicados_totales)

--- CONTEO DE VALORES NULOS ---
Legajo        0
Nombre        1
Comision      1
Nota_Final    2
dtype: int64

--- CONTEO DE DUPLICADOS ---
Filas exactamente duplicadas: 1


### Paso 3: Limpieza Fase 1 - Duplicados y Nulos Críticos
Si un sistema guardó la misma fila dos veces, no nos sirve. Si a un registro le falta el dato más importante (ej: el Nombre), probablemente debamos borrar esa fila entera.

In [ ]:
# 1. Eliminamos los duplicados exactos (el sistema retiene la primera aparición y borra la copia)
df_limpio = df_crudo.drop_duplicates()

# 2. Borramos las filas donde falte un dato fundamental (el Nombre en este caso)
# Usamos el parámetro 'subset' para decirle que solo mire esa columna
df_limpio = df_limpio.dropna(subset=['Nombre'])

print('--- DATASET SIN DUPLICADOS Y SIN NOMBRES NULOS ---')
display(df_limpio)

--- DATASET SIN DUPLICADOS Y SIN NOMBRES NULOS ---


,Legajo,Nombre,Comision,Nota_Final
0,TUP-01,Ana,A,8.5
1,TUP-02,Juan,A,NaN
2,TUP-03,Pedro,B,4.0
3,TUP-04,Laura,NaN,9.0
6,TUP-06,Carlos,C,6.0


### Paso 4: Limpieza Fase 2 - Imputación de datos (fillna)
No queremos borrar a Laura porque le falta la comisión, ni a Juan porque le falta la nota. Vamos a *imputar* (rellenar) esos valores con lógica de negocio.

In [ ]:
# Si falta la comisión, asumimos que está 'Pendiente de asignacion'
df_limpio['Comision'] = df_limpio['Comision'].fillna('Pendiente')
# df_limpio['Comision'].fillna('Pendiente', inplace=True)

# Si falta la Nota Final, podemos asumir que estuvo Ausente y le corresponde un 0
df_limpio['Nota_Final'] = df_limpio['Nota_Final'].fillna(0.0)

print('--- DATASET 100% LIMPIO Y LISTO PARA ANÁLISIS ---')
display(df_limpio)

# Verificación final
print('\nAuditoría Final de Nulos:')
print(df_limpio.isnull().sum())

--- DATASET 100% LIMPIO Y LISTO PARA ANÁLISIS ---


,Legajo,Nombre,Comision,Nota_Final
0,TUP-01,Ana,A,8.5
1,TUP-02,Juan,A,0.0
2,TUP-03,Pedro,B,4.0
3,TUP-04,Laura,Pendiente,9.0
6,TUP-06,Carlos,C,6.0



Auditoría Final de Nulos:
Legajo        0
Nombre        0
Comision      0
Nota_Final    0
dtype: int64


### El Desafío Final: ETL a Gran Escala (15.000 registros)
Vamos a simular que descargamos la base de datos de ingresantes de toda la universidad. Como pasa en la vida real, el sistema de inscripciones falló y la base de datos está sucia.
Primero, generamos el archivo simulando el caos.

In [ ]:
import pandas as pd
import numpy as np

# 1. Generamos los datos masivos
np.random.seed(42)
total_filas = 15000

carreras = ['Programacion', 'Redes', 'Sistemas', 'Ciberseguridad']

df_masivo = pd.DataFrame({
    'ID_Aspirante': np.arange(100000, 100000 + total_filas),
    'Nombre': ['Aspirante_' + str(i) for i in range(total_filas)],
    'Carrera': np.random.choice(carreras, total_filas),
    'Nota_Ingreso': np.random.uniform(1.0, 10.0, total_filas).round(1)
})

# 2. ENSUCIAMOS LOS DATOS A PROPÓSITO (El trabajo del analista)
# Insertamos 200 filas duplicadas
df_masivo = pd.concat([df_masivo, df_masivo.sample(200)], ignore_index=True)

# Insertamos 500 notas nulas (NaN) simulando ausentes
indices_nulos_notas = np.random.choice(df_masivo.index, 500, replace=False)
df_masivo.loc[indices_nulos_notas, 'Nota_Ingreso'] = np.nan

# Insertamos 50 nombres nulos (NaN) simulando errores graves de base de datos
indices_nulos_nombres = np.random.choice(df_masivo.index, 50, replace=False)
df_masivo.loc[indices_nulos_nombres, 'Nombre'] = np.nan

# Mezclamos las filas para que el desorden sea total y guardamos
df_masivo = df_masivo.sample(frac=1).reset_index(drop=True)
df_masivo.to_csv('ingresantes_sucio.csv', index=False)

print('¡Archivo ingresantes_sucio.csv generado con el caos incluído!')

¡Archivo ingresantes_sucio.csv generado con el caos incluído!


### Paso 1: Ingesta y Auditoría Médica
Recibimos el archivo crudo. Antes de hacer cualquier cálculo, pasamos el escáner para ver cuánta "basura" tenemos.

In [ ]:
df_ingreso = pd.read_csv('ingresantes_sucio.csv')

print('Nulos:')
print(df_ingreso.isnull().sum())

print('Duplicados:')
print(df_ingreso.duplicated().sum())

Nulos:
ID_Aspirante      0
Nombre           50
Carrera           0
Nota_Ingreso    500
dtype: int64
Duplicados:
178


### Paso 2: Fase de Limpieza (El corazón del ETL)
Aplicamos nuestras estrategias: Eliminamos lo inservible y rellenamos lo recuperable.

In [ ]:
print(f'Cantidad de dato originales: {len(df_ingreso)}')

df_limpio = df_ingreso.drop_duplicates()
# print(f'Cant reg: {len(df_limpio)}')

df_limpio = df_limpio.dropna(subset=['Nombre'])
# print(f'Cant reg: {len(df_limpio)}')

df_limpio['Nota_Ingreso'] = df_limpio['Nota_Ingreso'].fillna(0.0)
print(f'Cant reg: {len(df_limpio)}')

print(f'Nulos: {df_limpio.isnull().sum()}')

Cantidad de dato originales: 15200
Cant reg: 14972
Nulos: ID_Aspirante    0
Nombre          0
Carrera         0
Nota_Ingreso    0
dtype: int64


### Paso 3: Manipulación Avanzada (Índices y `loc`)
La base ya está limpia. Ahora convertimos el ID en nuestro Índice real para poder hacer cirugías de datos si hiciera falta.

In [ ]:
# df_limpio = df_limpio.set_index('ID_Aspirante')

display(df_limpio.head(5))

df_limpio.loc[111533, 'Nota_Ingreso'] = 8
display(df_limpio.loc[111533, :])

,Nombre,Carrera,Nota_Ingreso
ID_Aspirante,,,
112621,Aspirante_12621,Programacion,0.0
111533,Aspirante_11533,Ciberseguridad,8.0
103600,Aspirante_3600,Programacion,3.5
100164,Aspirante_164,Redes,4.2
103937,Aspirante_3937,Sistemas,9.4


,111533
Nombre,Aspirante_11533
Carrera,Ciberseguridad
Nota_Ingreso,8.0


### Paso 4: El Análisis Final (Filtros Múltiples)
El decano nos pide el listado de los "Ingresantes de Élite" de **Programación**: Alumnos de esa carrera que hayan sacado **más de 9** en el examen.

In [ ]:
mascara_prog = (df_limpio['Nota_Ingreso'] > 9) & (df_limpio['Carrera'] == 'Programacion')

df_prog = df_limpio[mascara_prog]

display(df_prog.head(5))

promedio = df_prog['Nota_Ingreso'].mean().round(2)
print(f'Promedio: {promedio}')

,Nombre,Carrera,Nota_Ingreso
ID_Aspirante,,,
101966,Aspirante_1966,Programacion,9.3
107904,Aspirante_7904,Programacion,9.4
107302,Aspirante_7302,Programacion,9.5
100278,Aspirante_278,Programacion,9.1
104498,Aspirante_4498,Programacion,9.4


Promedio: 9.53
